## SETUP -Import required libraries and define file paths.

In [1]:
# Step 0 — Setup

import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

print("Libraries imported successfully.")


Libraries imported successfully.


## Step 1 — Load Data



In [3]:
# Step 1 — Load Data

train_path = Path("train.csv")

df_train = pd.read_csv(train_path)

df_train.head()



,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Step 2 — Cleaning and Feature Engineering
Apply the same preprocessing steps created in the previous notebook.


In [5]:
# Step 2 — Cleaning and Feature Engineering

def extract_title(name):
    return name.split(",")[1].split(".")[0].strip()

def clean_data(df):
    df = df.copy()

    # Fill missing Age with median
    df["Age"] = df["Age"].fillna(df["Age"].median())

    # Fill missing Embarked with mode
    df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

    # Fill missing Fare (sometimes required)
    df["Fare"] = df["Fare"].fillna(df["Fare"].median())

    # Feature: Title
    df["Title"] = df["Name"].apply(extract_title)

    # Feature: Family Size
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

    # Feature: Deck (first letter of Cabin)
    df["Deck"] = df["Cabin"].astype(str).str[0]
    df["Deck"] = df["Deck"].replace("n", "U")  # Unknown

    # Drop columns used before
    drop_cols = ["Name", "Ticket", "Cabin", "PassengerId"]
    df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors="ignore")

    return df

# Apply cleaning to training set
train_clean = clean_data(df_train)

train_clean.head()


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Title,FamilySize,Deck
0,0,3,male,22.0,1,0,7.2500,S,Mr,2,U
1,1,1,female,38.0,1,0,71.2833,C,Mrs,2,C
2,1,3,female,26.0,0,0,7.9250,S,Miss,1,U
3,1,1,female,35.0,1,0,53.1000,S,Mrs,2,C
4,0,3,male,35.0,0,0,8.0500,S,Mr,1,U


## Step 3 — Encoding and Preprocessing
Apply Label Encoding to `Sex` and One-Hot Encoding to selected categorical features.


In [6]:
# Step 3 — Encoding and Preprocessing

# Label Encoding for Sex
le_sex = LabelEncoder()
train_clean["Sex"] = le_sex.fit_transform(train_clean["Sex"])

# Categorical columns for OneHotEncoder
categorical_cols = ["Embarked", "Title", "Deck"]

# ColumnTransformer with OneHotEncoder
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
    ],
    remainder="passthrough"
)

# Show the preprocessor setup
preprocessor


ColumnTransformer(remainder='passthrough',
                  transformers=[('cat', OneHotEncoder(handle_unknown='ignore'),
                                 ['Embarked', 'Title', 'Deck'])])

## Step 4 — Train the Final Random Forest Model
Train the optimized Random Forest using the best hyperparameters found in the previous notebook.


In [8]:
# Step 4 — Train the Final Random Forest Model

# Separate features and target
X = train_clean.drop(columns=["Survived"])
y = train_clean["Survived"]

# Best hyperparameters obtained from GridSearchCV
best_params = {
    "max_depth": 10,
    "min_samples_leaf": 2,
    "min_samples_split": 2,
    "n_estimators": 100,
    "random_state": 42
}

# Create the final model
model = RandomForestClassifier(**best_params)

# Full pipeline (preprocessing + model)
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])

# Train on the full training data
pipeline.fit(X, y)

print("Final model trained successfully with optimized hyperparameters.")
pipeline


Final model trained successfully with optimized hyperparameters.


C:\Users\Gustavo\anaconda3\Lib\site-packages\sklearn\compose\_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['Embarked', 'Title',
                                                   'Deck'])])),
                ('model',
                 RandomForestClassifier(max_depth=10, min_samples_leaf=2,
                                        random_state=42))])

## Step 5 — Predict on Test Set 
The instructor will provide the official `test.csv` file.
Once available, we will apply the same cleaning pipeline and generate the predictions.


In [10]:
# Step 5 — Predict on Test Set and Generate Final Submission

# Load the test set received from the instructor
test_path = Path("test.csv")
df_test = pd.read_csv(test_path)

# Keep PassengerId for submission
passenger_ids = df_test["PassengerId"]

# Apply the same cleaning as training data
test_clean = clean_data(df_test)

# Encode Sex using the same encoder
test_clean["Sex"] = le_sex.transform(test_clean["Sex"])

# Predict using the trained pipeline
test_predictions = pipeline.predict(test_clean)

# Create submission file
submission = pd.DataFrame({
    "PassengerId": passenger_ids,
    "Survived": test_predictions.astype(int)
})

submission.to_csv("submission.csv", index=False)
print("✔️ submission.csv saved successfully!")

submission.head()


✔️ submission.csv saved successfully!


,PassengerId,Survived
0,892,0
1,893,0
2,894,0
3,895,0
4,896,1
